# Preprocesamiento y Analisis Exploratorio — Retail Sales Dataset

**Grupo 2** — Angel Espin · Carlos Ramirez

---

## Que hace este cuaderno

Este notebook descarga los datos crudos, los limpia, crea nuevos atributos
y hace un analisis exploratorio (EDA) con graficos para entender la estructura
de los datos antes de pasarlos al dashboard de Streamlit.

**Importante:** este cuaderno NO incluye la visualizacion final (esa esta en
`streamlit_app.py`). Aca nos aseguramos de que los datos esten sanos y de
entender que patrones vale la pena graficar.

In [ ]:
# --- Carga de librerias ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Para que los graficos se vean bien en el notebook
%matplotlib inline
plt.rcParams["figure.figsize"] = (9, 4.5)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
plt.rcParams["font.size"] = 12

# Paleta de colores que vamos a usar en el dashboard (para mantener consistencia)
COL_CAT = {"Beauty": "#4E79A7", "Clothing": "#F28E2B", "Electronics": "#59A14F"}
COL_GEN = {"Female": "#E15759", "Male": "#4E79A7"}

BASE = Path.cwd()
RAW_URL = "https://raw.githubusercontent.com/RISHIshrivas/Retail-Sales-data-analysis/main/retail_sales_dataset.csv"
CLEAN = BASE / "retail_sales_clean.csv"

print("Listo")

---
## 1. Carga de datos

Descargamos el csv desde un espejo en GitHub para que esto funcione
sin tener que bajar archivos a mano. Convierte la columna Date a
formato datetime porque la necesitamos para derivar mes, trimestre, etc.

In [ ]:
df = pd.read_csv(RAW_URL)
df["Date"] = pd.to_datetime(df["Date"])

print("Dimensiones:", df.shape)
print("Columnas:", list(df.columns))
print()
df.head(10)

Vemos que el dataset tiene 1000 filas y 9 columnas. Cada fila es
una transaccion de venta. Los tipos de dato se ven correctos:
numeros para edad, cantidad, precio e importe; texto para genero
y categoria; fecha para Date.

---
## 2. Control de calidad

Antes de hacer cualquier analisis hay que verificar que los datos
estan completos y son consistentes. Revisamos:
- valores nulos
- filas duplicadas
- que los IDs sean unicos
- que Total Amount = Quantity * Price per Unit (consistencia aritmetica)
- el rango de fechas

In [ ]:
print("--- Nulos por columna ---")
print(df.isnull().sum())
print()

print("--- Duplicados ---")
print("Filas duplicadas:", df.duplicated().sum())
print()

print("--- Unicidad de IDs ---")
print("Transaction ID unicos:", df["Transaction ID"].nunique())
print("Customer ID unicos:", df["Customer ID"].nunique())
print()

# Chequeo aritmetico: cada Total Amount deberia ser Quantity * Price per Unit
inconsistentes = (df["Quantity"] * df["Price per Unit"] != df["Total Amount"]).sum()
print("--- Consistencia ---")
print("Filas donde Total != Quantity*Price:", inconsistentes)
print()

print("--- Rango temporal ---")
print("Fecha minima:", df["Date"].min().date())
print("Fecha maxima:", df["Date"].max().date())
print("Transacciones por anio:")
print(df["Date"].dt.year.value_counts().sort_index())

Los datos estan impecables: cero nulos, cero duplicados, consistencia
aritmetica perfecta. Casi todas las transacciones son de 2023 (998) y
hay 2 de enero 2024, que dejamos igual porque no afecta el analisis.

Los IDs de cliente son todos distintos, o sea que este dataset no tiene
clientes recurrentes. Eso limita el analisis "por cliente" pero igual
podemos ver patrones demograficos por edad y genero.

---
## 3. Exploracion de variables numericas

Veamos como se distribuyen las variables cuantitativas: Edad, Cantidad,
Precio Unitario e Importe Total. Esto nos da una idea de los rangos y
de si hay valores atipicos.

In [ ]:
print("--- Estadisticas descriptivas ---")
print(df[["Age", "Quantity", "Price per Unit", "Total Amount"]].describe().round(1))

In [ ]:
# Histograma de las variables numericas
fig, axes = plt.subplots(2, 2, figsize=(11, 7))
nombres = ["Age", "Quantity", "Price per Unit", "Total Amount"]
titulos = ["Distribucion de Edad", "Distribucion de Cantidad",
           "Distribucion de Precio Unitario", "Distribucion de Total Amount"]
for ax, col, tit in zip(axes.flat, nombres, titulos):
    ax.hist(df[col], bins=20, color="#4E79A7", edgecolor="white", linewidth=0.5)
    ax.set_title(tit)
    ax.set_xlabel(col)
    ax.set_ylabel("Frecuencia")
plt.tight_layout()
plt.show()

**Que vemos:**
- Edad: distribucion bastante uniforme entre 18 y 64, sin picos raros.
- Cantidad: valores entre 1 y 4, con mas frecuencia en 2 y 3 unidades.
- Precio Unitario: solo 5 valores distintos (25, 30, 50, 300, 500).
  Hay dos clusters claros: precios bajos (25-50) y altos (300-500).
- Total Amount: distribucion sesgada a la izquierda, refleja la mezcla
  de productos baratos en cantidad vs caros en poca cantidad.

Estos clusters en el precio nos sugieren crear una variable "NivelPrecio"
para agrupar productos economicos y premium.

---
## 4. Exploracion de variables categoricas

Veamos la distribucion de Genero y Categoria de Producto, y como se
relacionan con los ingresos.

In [ ]:
print("--- Frecuencia de Genero ---")
print(df["Gender"].value_counts())
print()
print("--- Frecuencia de Categoria ---")
print(df["Product Category"].value_counts())
print()

# Grafico de barras: ingresos totales por categoria
cat_ing = df.groupby("Product Category")["Total Amount"].sum().sort_values()
print("--- Ingresos por categoria ---")
print(cat_ing.apply(lambda x: f"${x:,.0f}"))

In [ ]:
# Dos graficos lado a lado: ingresos y unidades por categoria
cat_stats = df.groupby("Product Category").agg(
    Ingresos=("Total Amount", "sum"),
    Unidades=("Quantity", "sum"),
    Ticket=("Total Amount", "mean")
).sort_values("Ingresos")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
colores = [COL_CAT[c] for c in cat_stats.index]

ax1.barh(cat_stats.index, cat_stats["Ingresos"], color=colores)
ax1.set_title("Ingresos por Categoria")
ax1.set_xlabel("Ingresos ($)")

unids = cat_stats.sort_values("Unidades")
colores2 = [COL_CAT[c] for c in unids.index]
ax2.barh(unids.index, unids["Unidades"], color=colores2)
ax2.set_title("Unidades Vendidas por Categoria")
ax2.set_xlabel("Unidades")

plt.tight_layout()
plt.show()

print("Ticket promedio por categoria:")
print(cat_stats["Ticket"].apply(lambda x: f"${x:,.2f}"))

Electronics y Clothing estan muy parejas en ingresos (156K vs 155K), mientras
Beauty queda atras con 143K. Pero en unidades pasa algo distinto: Clothing
vende mas unidades, Beauty le sigue, y Electronics vende menos. Esto tiene
sentido porque Electronics tiene precios mas altos, entonces con menos
unidades igual genera buenos ingresos.

El ticket promedio lo confirma: Electronics tiene el ticket mas alto ($458),
Beauty el mas bajo ($448) y Clothing en el medio ($455). Las diferencias
son chicas porque los precios base son los mismos para todas las categorias.

---
## 5. Tendencia temporal

El dashboard tiene como objetivo mostrar como evolucionan las ventas
en el tiempo. Vamos a ver la tendencia mensual de ingresos.

In [ ]:
df["MesNombre"] = df["Date"].dt.month.apply(
    lambda m: ["Enero","Febrero","Marzo","Abril","Mayo","Junio",
              "Julio","Agosto","Septiembre","Octubre","Noviembre","Diciembre"][m-1]
)
serie_mensual = df.groupby("MesNombre")["Total Amount"].sum()
orden = ["Enero","Febrero","Marzo","Abril","Mayo","Junio",
         "Julio","Agosto","Septiembre","Octubre","Noviembre","Diciembre"]

# Nos aseguramos de que los meses esten en orden cronologico
serie_mensual = serie_mensual.reindex(orden)

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(serie_mensual.index, serie_mensual.values, marker="o",
        color="#4E79A7", linewidth=2, markersize=8)
ax.set_title("Ingresos Mensuales (2023)")
ax.set_xlabel("Mes")
ax.set_ylabel("Ingresos ($)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

print("Mejor mes:", serie_mensual.idxmax(), f"(${serie_mensual.max():,.0f})")
print("Peor mes:", serie_mensual.idxmin(), f"(${serie_mensual.min():,.0f})")
print("Promedio mensual:", f"${serie_mensual.mean():,.0f}")

Se ve bastante variacion entre meses. Mayo es el mejor mes con $53K,
mientras Septiembre es el mas bajo con $24K. Esto nos dice que hay
estacionalidad y vale la pena explorarla en el dashboard.

El promedio mensual es de $38K, y varios meses se desvian bastante
de ese valor. Seria util agregar una linea de promedio en el grafico
del dashboard para tener contexto.

In [ ]:
# Tendencia por categoria (lineas separadas)
cat_mes = df.groupby(["MesNombre", "Product Category"])["Total Amount"].sum().reset_index()

fig, ax = plt.subplots(figsize=(10, 4.5))
for cat in ["Beauty", "Clothing", "Electronics"]:
    sub = cat_mes[cat_mes["Product Category"] == cat]
    # Reordenar por mes
    sub = sub.set_index("MesNombre").reindex(orden).reset_index()
    ax.plot(sub["MesNombre"], sub["Total Amount"], marker="o",
            label=cat, color=COL_CAT[cat], linewidth=2)
ax.set_title("Ingresos Mensuales por Categoria")
ax.set_xlabel("Mes")
ax.set_ylabel("Ingresos ($)")
ax.legend()
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

Cada categoria tiene su propio patron. Clothing parece mas estable,
mientras Beauty y Electronics tienen picos y valles mas marcados.
Las tres categorias comparten el valle de Septiembre, lo que sugiere
que es un mes debil para toda la tienda, no solo para un producto.

---
## 6. Perfil demografico

Veamos como se distribuyen los ingresos por edad y genero.
Esto ayuda a entender a que segmentos de clientes apuntar.

In [ ]:
# Creamos grupos de edad para el analisis
df["GrupoEdad"] = pd.cut(df["Age"], bins=[18, 25, 35, 45, 55, 65],
                          labels=["18-25", "26-35", "36-45", "46-55", "56-65"],
                          include_lowest=True, right=True)

piv = df.pivot_table(index="GrupoEdad", columns="Gender",
                     values="Total Amount", aggfunc="sum")

print("--- Ingresos por Grupo de Edad y Genero ---")
print(piv.applymap(lambda x: f"${x:,.0f}"))

In [ ]:
# Grafico de barras agrupadas
x = np.arange(len(piv.index))
w = 0.35

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.bar(x - w/2, piv["Female"], w, label="Female", color=COL_GEN["Female"])
ax.bar(x + w/2, piv["Male"], w, label="Male", color=COL_GEN["Male"])
ax.set_xticks(x)
ax.set_xticklabels(piv.index)
ax.set_title("Ingresos por Grupo de Edad y Genero")
ax.set_xlabel("Grupo de Edad")
ax.set_ylabel("Ingresos ($)")
ax.legend()
plt.tight_layout()
plt.show()

El grupo 26-35 lidera en ingresos totales, seguido de cerca por 46-55 y 36-45.
Los grupos de los extremos (18-25 y 56-65) son los que menos gastan.

En cuanto a genero: en 18-25 y 26-35, las mujeres gastan mas que los hombres.
En 36-45 estan casi iguales. En 46-55 y 56-65 los hombres gastan un poco mas.
Son diferencias chicas, pero pueden ser utiles para segmentar campañas.

---
## 7. Relacion entre precio y cantidad

Otra dimension interesante: como se relaciona el precio unitario con
la cantidad comprada. Esto revela patrones de compra.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for cat in ["Beauty", "Clothing", "Electronics"]:
    sub = df[df["Product Category"] == cat]
    # Agregamos un poco de ruido para ver puntos superpuestos
    jitter = np.random.normal(0, 0.05, len(sub))
    ax.scatter(sub["Quantity"] + jitter, sub["Price per Unit"],
               alpha=0.5, label=cat, color=COL_CAT[cat], s=30)
ax.set_xlabel("Cantidad")
ax.set_ylabel("Precio por Unidad ($)")
ax.set_title("Precio vs Cantidad por Categoria")
ax.legend()
ax.set_xticks([1, 2, 3, 4])
plt.tight_layout()
plt.show()

Se ven tres clusters claros:
- Belleza: precio bajo ($25-$50), se compra en cantidades de 2 a 4 unidades
- Electronica: precio alto ($300-$500), casi siempre 1 o 2 unidades
- Ropa: combinacion de ambos extremos, tanto compras de 1 unidad cara
  como varias unidades economicas

Esto confirma que vale la pena incluir un grafico de dispersion en el
dashboard para mostrar estos patrones.

---
## 8. Analisis por dia de la semana

Para completar el analisis temporal, veamos como se distribuyen
los ingresos segun el dia de la semana.

In [ ]:
dias = {0:"Lunes", 1:"Martes", 2:"Miercoles", 3:"Jueves",
        4:"Viernes", 5:"Sabado", 6:"Domingo"}
orden_dias = ["Lunes", "Martes", "Miercoles", "Jueves", "Viernes", "Sabado", "Domingo"]

df["DiaSemana"] = df["Date"].dt.dayofweek.map(dias)
ing_dia = df.groupby("DiaSemana")["Total Amount"].sum().reindex(orden_dias)
trans_dia = df.groupby("DiaSemana")["Transaction ID"].count().reindex(orden_dias)

fig, ax1 = plt.subplots(figsize=(9, 4.5))
color_bar = "#4E79A7"
ax1.bar(ing_dia.index, ing_dia.values, color=color_bar, alpha=0.8, label="Ingresos")
ax1.set_ylabel("Ingresos ($)", color=color_bar)
ax1.tick_params(axis="y", labelcolor=color_bar)

ax2 = ax1.twinx()
color_line = "#E15759"
ax2.plot(trans_dia.index, trans_dia.values, marker="o",
         color=color_line, linewidth=2, label="Transacciones")
ax2.set_ylabel("Transacciones", color=color_line)
ax2.tick_params(axis="y", labelcolor=color_line)

ax1.set_title("Ingresos y Transacciones por Dia de la Semana")
fig.tight_layout()
plt.show()

print("Ingresos:")
print(ing_dia.apply(lambda x: f"${x:,.0f}"))
print()
print("Transacciones:")
print(trans_dia)

Viernes y Sabado son los dias mas fuertes. Los lunes tambien tienen
buen rendimiento. Miercoles y Jueves son los mas debiles.

El patron de transacciones sigue mas o menos el mismo que el de ingresos,
aunque no exactamente: por ejemplo, Sabado tiene menos transacciones que
Viernes pero ingresos similares, lo que sugiere que el Sabado se venden
productos mas caros por transaccion.

---
## 9. Derivacion de atributos y exportacion

Con base en el EDA, creamos los atributos derivados que vamos a usar
en el dashboard. La mayoria salen de la fecha (mes, trimestre, dia de
semana) y de agrupar edad y precio en categorias.

In [ ]:
MESES_SORT = {
    1:"01-Enero", 2:"02-Febrero", 3:"03-Marzo", 4:"04-Abril",
    5:"05-Mayo", 6:"06-Junio", 7:"07-Julio", 8:"08-Agosto",
    9:"09-Septiembre", 10:"10-Octubre", 11:"11-Noviembre", 12:"12-Diciembre"
}
DIAS_SORT = {
    0:"1-Lunes", 1:"2-Martes", 2:"3-Miercoles", 3:"4-Jueves",
    4:"5-Viernes", 5:"6-Sabado", 6:"7-Domingo"
}

# Atributos temporales
df["Anio"] = df["Date"].dt.year
df["Mes"] = df["Date"].dt.month
df["NombreMes"] = df["Mes"].map(MESES_SORT)
df["Trimestre"] = "T" + df["Date"].dt.quarter.astype(str)
df["DiaSemana"] = df["Date"].dt.dayofweek.map(DIAS_SORT)
df["TipoDia"] = df["Date"].dt.dayofweek.apply(
    lambda d: "Fin de semana" if d >= 5 else "Entre semana"
)

# Grupos de edad (vimos en el EDA que los rangos de 10 anos funcionan bien)
df["GrupoEdad"] = pd.cut(df["Age"],
    bins=[18, 25, 35, 45, 55, 65],
    labels=["18-25", "26-35", "36-45", "46-55", "56-65"],
    include_lowest=True, right=True
).astype(str)

# Nivel de precio: los histogramas mostraron dos clusters naturales
df["NivelPrecio"] = df["Price per Unit"].apply(
    lambda p: "Bajo (<=$50)" if p <= 50 else "Alto (>=$300)"
)

print("Atributos finales (", len(df.columns), "):")
print(list(df.columns))
print()
print("Primeras filas del dataset limpio:")
df.head()

---
## 10. Resumen y resumen final

Ultimo chequeo antes de exportar: confirmamos los totales generales
y que no haya perdido filas en el proceso.

In [ ]:
print("=" * 55)
print("RESUMEN DEL DATASET LIMPIO")
print("=" * 55)
print(f"{ 'Filas:':<25} {len(df):>10,}")
print(f"{ 'Columnas:':<25} {len(df.columns):>10}")
print(f"{ 'Ingresos totales:':<25} ${df['Total Amount'].sum():>8,.0f}")
print(f"{ 'Ticket promedio:':<25} ${df['Total Amount'].mean():>8,.2f}")
print(f"{ 'Unidades vendidas:':<25} {df['Quantity'].sum():>10,}")
print(f"{ 'Categorias:':<25} {df['Product Category'].nunique():>10}")
print(f"{ 'Rango de edad:':<25} {df['Age'].min()} - {df['Age'].max()}")
print(f"{ 'Rango de fechas:':<25} {df['Date'].min().date()} a {df['Date'].max().date()}")
print(f"{ 'Valores nulos:':<25} {df.isnull().sum().sum():>10}")
print(f"{ 'Filas duplicadas:':<25} {df.duplicated().sum():>10}")

# Exportamos
df.to_csv(CLEAN, index=False, encoding="utf-8-sig")
print()
print("CSV exportado a:", CLEAN)

---
## Lo que aprendimos (y que vamos a graficar en el dashboard)

Despues de este analisis, sabemos que el dashboard deberia incluir:

1. **KPIs** - ingresos totales, transacciones, ticket promedio, unidades
2. **Tendencia mensual** - con linea de promedio de referencia
3. **Desglose por categoria** - linea multiple o area apilada
4. **Ingresos por categoria** - barras ordenadas
5. **Perfil demografico** - barras agrupadas edad x genero
6. **Precio vs Cantidad** - grafico de dispersion (burbujas)
7. **Distribucion temporal** - por mes y por dia de la semana

Todo esto se implementa en `streamlit_app.py` usando los datos limpios
que acabamos de generar.